> 闭包（closure）是“一个函数对象”，它**引用了来自外层作用域的自由变量**，并且这些变量在函数定义的作用域已经结束后仍然存在。

> 闭包捕获的是变量的“绑定（Binding）”或“引用（Reference）”，而不是在创建闭包那一瞬间变量的“快照值（Value）”。在 Python 中，这个现象被称为延迟绑定（Late Binding）。闭包在被调用时，才会去外部作用域查找变量的值。

### 必要性

- 每个生成器都有自己的私有状态（必要性：隔离状态 + 隐藏实现）
- nonlocal vs. global
    - nonlocal 和 global 都是作用域声明：告诉解释器“这个名字的赋值，不要当成当前函数的局部变量”。
    - global：指向“模块级全局作用域”
    - nonlocal：指向“最近一层外部函数作用域”的变量

In [9]:
def make_id_generator(prefix):
    n = 0
    def next_id():
        nonlocal n
        n += 1
        return f"{prefix}{n:04d}"
    return next_id

user_id  = make_id_generator("U")
order_id = make_id_generator("O")

print(user_id(), user_id(), user_id())     # U0001 U0002 U0003
print(order_id(), order_id())              # O0001 O0002
print(user_id())                           # U0004

U0001 U0002 U0003
O0001 O0002
U0004


In [11]:
# 不用闭包的“常见替代”：全局变量会互相干扰（这就是闭包解决的问题）
n = 0
def next_id(prefix):
    global n
    n += 1
    return f"{prefix}{n:04d}"

print(next_id("U"), next_id("U"))   # U0001 U0002
print(next_id("O"))                 # O0003  <- 订单编号被用户编号“污染”

U0001 U0002
O0003


### 易错点

In [5]:
def make_funcs_wrong():
    funcs = []
    for i in range(3):
        funcs.append(lambda: i)   # 捕获的是变量 i 的绑定
    return funcs

fs = make_funcs_wrong()
print([f() for f in fs])  # 你可能期望 [0,1,2]，实际是？

[2, 2, 2]


In [6]:
fs[0].__closure__

(<cell at 0x7c129809beb0: int object at 0x59feecb84208>,)

### 利用默认参数传递“值” (最常用)

> 不再是闭包了

In [7]:
def make_funcs_right_default():
    funcs = []
    for i in range(3):
        funcs.append(lambda i=i: i)  # 把当时的值绑定到默认参数
    return funcs

fs = make_funcs_right_default()
print([f() for f in fs])  # [0, 1, 2]

[0, 1, 2]


In [8]:
fs[0].__closure__